# Create Per-SDG Scopus Splits

This notebook loads the raw Scopus SDG exports, cleans abstract text, and writes per-SDG 80/20 train/test CSV files for downstream labeling runs.


In [ ]:
import time
import urllib
from pathlib import Path
from typing import Dict, List

import jupyter_black
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
import requests
import seaborn as sns
from dotenv import load_dotenv
# from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm
from transformers import BertTokenizer
import Levenshtein as lev

# Use verbose pandas mode
pd.options.display.max_rows = 200

load_dotenv()
jupyter_black.load()

In [ ]:
def clean_abstracts(df):
    df_cleaned = df.copy()

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.strip()

    # for _, row in df_cleaned.iterrows():
    #     if "Success in marriage markets has lasting" in row["Abstract"]:
    #         print(f"Found it: {row['Abstract']}")

    # First, the specific case where there is no punctuation or space between the year and the first letter of the abstract
    copyright_regex = r"(© \d{4})([A-Z])"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
        lambda abstract: re.sub(
            copyright_regex, lambda match: match.group(2), abstract
        ).strip()
    )

    # Next, handle cases where the end of the copyright statement and the beginning of the next sentence are merged without a space.
    copyright_regex = r"^((?:Copyright ©|©) (?:\d{4})?(?:[[:alpha:][:punct:]\s][^\.]+?)[A-Za-z)]([A-Z][a-z\s\d[:punct:]]))"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
        lambda abstract: re.sub(
            copyright_regex, lambda match: match.group(2), abstract
        ).strip()
    )

    # Then, remove specific strings that are known to be present in the abstracts
    copyright_regexes = [
        r"^(?:Copyright ©|©) (\d{4}) Elsevier B\.V\.",
        r"^© Elsevier B\.V\.",
        r"© \d{4} Elsevier Ltd",
        r"© \d{4} Published by Elsevier Ltd\.",
        r"Published by Elsevier Ltd\.",
        r"S\. Government work and not under copyright protection in the US; foreign copyright protection may apply\.",
        r"^© The Author(s) \d{4}\.",
        r"^© \d{4} The author\(s\)\.",
        r"^© \d{4} The authors\.",
        r"^© \d{4} The author\.",
        r"^© \d{4}, The Author\(s\)\.",
        r"^© \d{4} The Author\(s\)",
        r"^© \d{4} The Author",
        r"^© \d{4} The Authors",
        r"\(Figure presented\.\)© The Author\(s\) \d{4}\.",
        r"ICIC International © \d{4}\.",
        r"^©2024 National Information and Documentation Center \(NIDOC\)",
        r"^© E. Green and F. Ritchie.",
        r"^@ \d{4} China University of Geosciences \(Beijing\) and Peking University",
        r"^© \d{4} Akademikerförbundet SSR \(ASSR\) and John Wiley \& Sons Ltd\.",
        r"© \d{4} Society of Chemical Industry\.",
        r"^© by the author, licensee University of Lodz – Lodz University Press, Lodz, Poland\.",
        r"This is an Open Access article under the CC BY(?:-NC-ND|-SA)? 4\.0 license\.?",
        r"This is an open access article under the CC BY-SA license\.?",
        r"This is an open access article under the CC BY-NC License \(http://creativecommons.org/licenses/by-nc/4\.0/\)\.",
        r"This is an open access article under the CC BY-SA https://creativecommons.org/licenses/by-sa/4\.0/\.",
        r"© This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4\.0/)\.",
        r"This is an Open Access article distributed under the terms of the Creative Commons Attribution License, which permits unrestricted use, distribution, and reproduction in any medium, provided the original work is properly cited\.",
        r"This is an open access article distributed under the terms of the Creative Commons Attribution 4\.0 International License \(https://creativecommons.org/licenses/by/4\.0/\), allowing third parties to copy and redistribute the material in any medium or format and to remix, transform, and build upon the material for any purpose, even commercially, provided the original work is properly cited and states its license\.",
        r"Published by Wolters Kluwer Health, Inc\.",
        r"Journal of The Science of Food and Agriculture published by John Wiley \& Sons Ltd on behalf of Society of Chemical Industry\.",
        r"International Transactions in Operational Research © 2022 International Federation of Operational Research Societies\.",
        r"(^©.*?All rights reserved\.?)",
        r"(^Copyright.*?All rights reserved\.?)",
        r"^© \d{4} Copyright:",
        r"© \d{4} American Medical Association\.",
        r"©\d{4} American Medical Association\.",
        r"© \d{4} AMA\.",
        r"© \d{4} \w+ \w+ \w+\.",
        r"© \d{4}, \w+ \w+ \w+\.",
        r"© \d{4} \w+ \w+ \w+ \w+\. All rights reserved."
        r"© \d{4} \w+ \w+\."
        r"All rights reserved\.",
        r"© \d{4}, \w+ \w+\." r"All rights reserved\.",
        r"© \d{4} American Medical Association All rights reserved\."
        r"Copyright \d{4} American Medical Association\. All rights reserved\.",
        r"Copyright © \d{4} JAMA - Journal of the American Medical Association\. All rights reserved\.",
        r"© \d{4} JAMA - Journal of the American Medical Association\. All rights reserved\.",
        r"© JAMA - Journal of the American Medical Association \d{4}\.",
        r"© \d{4} National Academy of Sciences\. All rights reserved\.",
        r"© \d{4} The Authors",
        r"© The Author\(s\) \d{4}\.",
        r"© \d{4} WILEY-VCH Verlag GmbH & Co\. KGaA, Weinheim",
        r"© \d{4}, Published with license by Taylor & Francis\.",
        r"© \d{4} Financial Management Association International",
        r"\[copyright information to be updated in production process\]\.",
    ]

    for regex in copyright_regexes:
        df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
            lambda x: re.sub(regex, "", x, flags=re.IGNORECASE).strip()
        )

    # Next, handle other weird cases.
    copyright_regex = r"^© 2023‘"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.replace(
        copyright_regex, r"‘", regex=True
    )

    copyright_regex = r"^© 20231\-MeV"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.replace(
        copyright_regex, r"1-MeV", regex=True
    )

    copyright_regex = r"^© 20233D"

    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.replace(
        copyright_regex, r"3D", regex=True
    )

    # Do these last.
    copyright_regexes = [
        r"^©\s?(?:\d{4})?.*?\.",
        r"(^Copyright\s*©.*?\.)",
        r"(^Copyright\s*:\s*©.*?\.)",
        r"© \d{4} American Society of Civil Engineers\.",
        r"Global Change Biology© \d{4} The Authors\.",
        r"© \d{4},? The Author\(s\)\.",
        r"© \d{4},? The Authors.",
    ]

    for regex in copyright_regexes:
        df_cleaned["Abstract"] = df_cleaned["Abstract"].apply(
            lambda x: re.sub(regex, "", x, flags=re.IGNORECASE).strip()
        )

    # Optionally, strip any leading/trailing whitespace that might be left after the replacement
    df_cleaned["Abstract"] = df_cleaned["Abstract"].str.strip()
    # Reset the index after dropping rows
    df_cleaned.reset_index(drop=True, inplace=True)

    return df_cleaned

In [ ]:
# Read and combine all dataframes; clean the data
dataframes = []

data_path = Path("data")
for i in range(1, 18):
    file_path = data_path / "raw" / "scopus" / f"SDG{i:02}.csv"  # Constructs file path
    df = pd.read_csv(file_path)

    df = df[
        ~df["Abstract"].isin(["[No abstract available]", "None", ""])
        & df["Title"].notna()
        & df["Abstract"].notna()
    ].copy()

    # Reset the index after dropping rows
    df.reset_index(drop=True, inplace=True)

    # Clean abstracts
    df = clean_abstracts(df)

    df["SDG"] = i

    row_count = len(df)
    print(f"SDG {i} - Row count: {row_count}")

    dataframes.append(df)

In [ ]:
for i, df in enumerate(dataframes):
    count = 0
    for abstract in df["Abstract"]:
        if (
            "copyright" in abstract.lower()
            or "©" in abstract
            or "open access article" in abstract.lower()
        ):
            print(f"{i}: {abstract}")
            count += 1
        if count > 2:
            break

In [ ]:
dataframes[0].head(2)

In [ ]:
columns_to_keep = ["DOI", "Abstract", "SDG"]

for i, df in enumerate(dataframes):
    dataframes[i] = df[columns_to_keep]

dataframes[0].head(2)

In [ ]:
for i, df in enumerate(dataframes):
    # Features (all columns except SDG)
    X = df.drop(columns=["SDG"])
    y = df["SDG"]  # Labels (single integer SDG column)

    # Stratified split (single-label classification)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=200)

    for train_index, test_index in sss.split(X, y):
        train_df = df.iloc[train_index].reset_index(drop=True)
        test_df = df.iloc[test_index].reset_index(drop=True)

    # Format file paths correctly
    train_file_path = data_path / "processed" / f"sdg{i+1}_2023_train.csv"
    test_file_path = data_path / "processed" / f"sdg{i+1}_2023_test.csv"

    # Write to CSV
    train_df.to_csv(train_file_path, index=False)
    test_df.to_csv(test_file_path, index=False)

    print(f"Saved: {train_file_path} and {test_file_path}")
    print(f"Train: {train_df.shape}, Test: {test_df.shape}")